# BrainGuard: final model (ResNet50) on a free Colab GPU

The architecture screen showed that **ResNet50** beats the pre-specified EfficientNetB0 on the same patient folds. This notebook makes it the final model:

1. retrains ResNet50 on **exactly the same patient-grouped 5 folds**, this time saving each fold model;
2. re-runs the **robustness tests** (blur, noise, contrast, resolution, JPEG, rotation ...) on those models;
3. re-runs **SHAP (expected gradients) and Grad-CAM**, scored against the radiologists' tumour masks, with the randomisation sanity check.

**Before you start:** *Runtime → Change runtime type → T4 GPU → Save*, then *Runtime → Run all*.

- Total time is about **1–1.5 hours**: about 10 minutes of downloads, 5 minutes of smoke test, then about 30 minutes of training, 10 of robustness and 20–30 of explanations.
- Progress is saved to Google Drive (`MyDrive/brainguard_final`, about 1–2 GB for the five fold models). **If Colab disconnects, run all cells again**: finished folds and finished stages are skipped.
- At the end, `final_model_bundle.zip` (predictions and explanation scores, a few MB) downloads automatically. Upload it to the chat.

In [7]:
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit('NO GPU. Runtime -> Change runtime type -> T4 GPU -> Save, then Runtime -> Run all. '
                     'If T4 is unavailable, the free daily GPU quota is used up: try again later.')
print('GPU:', gpu.stdout.strip())

GPU: Tesla T4, 15360 MiB


In [8]:
import os
WORKDIR = '/content/brainguard_final'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKDIR = '/content/drive/MyDrive/brainguard_final'
    print('Google Drive connected - progress will be saved and can resume.')
except Exception as e:
    print('Google Drive not connected (', type(e).__name__, ') - continuing without it; keep this tab open.')
os.makedirs(WORKDIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive connected - progress will be saved and can resume.


In [9]:
%cd /content
!rm -rf BrainGuard-MRI-Tumor-Detection
!git clone -q https://github.com/Muneera-DSA/BrainGuard-MRI-Tumor-Detection.git
%cd /content/BrainGuard-MRI-Tumor-Detection
!pip install -q -e ".[dl]"
import sys
sys.path.insert(0, '/content/BrainGuard-MRI-Tumor-Detection/src')   # make the package importable in this session without a restart

/content
/content/BrainGuard-MRI-Tumor-Detection
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for brainguard (pyproject.toml) ... done


## 1. Download and convert the figshare data (about 900 MB, 5–10 minutes)

In [10]:
from pathlib import Path
from brainguard import figshare
raw, fig = Path('/content/data/figshare_raw'), Path('/content/data/figshare')
if not (fig / 'metadata.csv').exists():
    figshare.download(raw)
    figshare.convert(raw, fig)
print(len(list((fig / 'images').glob('*'))), 'images ready')

3064 images ready


## 2. Smoke test: train 2 folds × 3 epochs, save, reload, robustness and explanations on a few images (about 5 minutes). Stops here if anything is wrong.

In [ ]:
%%writefile /content/smoke_check.py
# Runs in its own process so that it releases the GPU before the full run.
import pandas as pd
from pathlib import Path
from brainguard import explain, robustness
pred = pd.read_csv('/content/smoke/predictions/resnet50_final.csv', dtype={'pid': str})
pred['abs_path'] = [f'/content/data/figshare/images/{i}.png' for i in pred.idx]
small = pred.groupby('fold').head(40)
rob = robustness.run(Path('/content/smoke/models/resnet50_final'), small, 224, 32,
                     {k: robustness.PERTURBATIONS[k] for k in list(robustness.PERTURBATIONS)[:3]})
ex = explain.run(small, Path('/content/data/figshare'), Path('/content/smoke/models/resnet50_final'), Path('/content/smoke/explain'),
                 224, n_images=6, n_background=8, n_samples=16, n_sanity=2)
# A 3-epoch model can give blank Grad-CAM maps (score 0/0 = NaN), so only the plumbing is checked here.
assert len(rob) == 3 * len(small), f'robustness rows {len(rob)}'
assert len(ex) == 6 and ex.shap_energy_in_mask.notna().any(), 'explanations missing'
assert Path('/content/smoke/explain/explain_sanity.csv').exists(), 'sanity check missing'
print(f'Grad-CAM maps with signal: {ex.cam_energy_in_mask.notna().sum()} of {len(ex)} (blank maps are normal for a 3-epoch model)')
print('SMOKE TEST PASSED: training, saving, reloading, robustness and explanations all work.')

In [ ]:
!rm -rf /content/smoke
!python -m brainguard.train --experiments resnet50_final --figshare-dir /content/data/figshare --artifacts-dir /content/smoke --quick 2>&1 | grep -E "fold|Error|Traceback|GPU"
out = !python /content/smoke_check.py 2>&1
print('\n'.join(line for line in out if not line.startswith(('W0000', 'I0000'))))
assert any('SMOKE TEST PASSED' in line for line in out), 'Smoke test failed: copy the output above and send it back'

## 3. Train the final model: ResNet50 × 5 folds, models saved (about 30 minutes)

In [13]:
!python -m brainguard.train --experiments resnet50_final --figshare-dir /content/data/figshare --artifacts-dir "{WORKDIR}/artifacts" 2>&1 | grep -E "Epoch|fold|Error|Traceback|GPU"

W0000 00:00:1791880771.125355    7875 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1791880771.127210    7875 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
GPU found: mixed precision on
Epoch 1/5
Epoch 2/5
Epoch 3/5
Epoch 4/5
Epoch 5/5
Epoch 1/20
Epoch 2/20
Epoch 3/20
Epoch 4/20
Epoch 5/20
Epoch 6/20
Epoch 7/20
Epoch 8/20
Epoch 9/20
[resnet50_final] fold 0: acc 0.933 in 179s
Epoch 1/5
Epoch 2/5
Epoch 3/5
Epoch 4/5
Epoch 5/5
Epoch 1/20
Epoch 2/20
Epoch 3/20
Epoch 4/20
Epoch 5/20
Epoch 6/20
Epoch 7/20
Epoch 8/20
[resnet50_final] fold 1: acc 0.886 in 261s
Epoch 1/5
Epoch 2/5
Epoch 3/5
Epoch 4/5
Epoch 5/5
Epoch 1/20
Epoch 2/20
Epoch 3/20
Epoch 4/20
Epoch 5/20
Epoch 6/20
Epoch 7/20
Epoch 8/20
[resnet50_final] fold 2: acc 0.924 in 250s
Ep

## 4. Robustness: every fold model re-scores its own unseen patients under each perturbation (about 10 minutes)

In [14]:
!python -m brainguard.robustness --figshare-dir /content/data/figshare --artifacts-dir "{WORKDIR}/artifacts" 2>&1 | grep -v -E "^W0000|^I0000"

To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 70 variables whereas the saved optimizer has 74 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


## 5. Explanations: SHAP + Grad-CAM on 150 unseen images, scored against tumour masks, with the sanity check (about 20–30 minutes)

*In the published run, this step and step 6 were run from one standalone cell after step 4 (same commands, same saved fold models). Their outputs are the files in `results/explain/resnet50_final/` and `reports/figures/explain_examples.png`.*

In [ ]:
!python -m brainguard.explain --figshare-dir /content/data/figshare --artifacts-dir "{WORKDIR}/artifacts" --reports-dir "{WORKDIR}/reports" 2>&1 | grep -v -E "^W0000|^I0000"

## 6. Download `final_model_bundle.zip` and upload it to the chat

In [ ]:
# Self-contained, so it also works after a Colab restart
import os, zipfile
from pathlib import Path
WORKDIR = '/content/drive/MyDrive/brainguard_final' if os.path.isdir('/content/drive/MyDrive') else '/content/brainguard_final'
art = Path(WORKDIR) / 'artifacts'
name = 'resnet50_final'
files_ = [art / 'predictions' / f'{name}.csv', art / 'predictions' / f'robustness_{name}.csv',
          *sorted((art / 'predictions' / name).glob('fold*')), *sorted((art / 'explain' / name).glob('*')),
          Path(WORKDIR) / 'reports' / 'figures' / 'explain_examples.png']
print('folds trained:', len(list((art / 'predictions' / name).glob('fold*_history.json'))), 'of 5')
for f in files_:
    print(('OK      ' if f.exists() else 'MISSING ') + str(f.relative_to(WORKDIR)))
out = Path(WORKDIR) / 'final_model_bundle.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files_:
        if f.exists():
            z.write(f, str(f.relative_to(WORKDIR)))
print(out, round(out.stat().st_size / 1e6, 2), 'MB')
from google.colab import files
files.download(str(out))